# MEAN: direct the work, check the result
### Working with LLMs | Pilot v0.1 | 25 September 2026

**Audience:** SAS professionals learning a checkable SAS-to-Python collaboration workflow.

**Outcome:** Explain the required behavior, request a bounded translation, and justify an acceptance decision from visible checks and stated limits.

> We are reducing unnecessary choices and building reproducible checks. We are not promising identical model responses.

**This is a self-contained review workbook.** It includes an authored teaching reference and locally recorded Python results. It contains no Bedrock API connection, no captured response from a live model, and no SAS execution evidence. The SAS and live-model portions remain for the approved pilot environment.

**Two ways to use it:** Review the supplied example without contacting a model, or make a working copy and replace the designated candidate function after obtaining and inspecting a response through the approved portal. The companion HTML is read-only.

## Before you run anything

Read the code cells first. Use the approved Python environment with pandas installed; this notebook does not install packages. A code cell executes in its selected kernel, while the fenced SAS below is reference text, not SAS execution. [Jupyter]

For a first review, execute the supplied cells from top to bottom. Before finalizing a run, restart the kernel and run all reviewed cells again. Restarting does not delete workspace files. Save only approved inputs and outputs.

For a controlled A/B exercise, obtain both first responses in separate fresh conversations **before** showing the reference solution. Keep source, starting instructions, visible settings, and access tools the same. A successful A response is a valid observation, not a failed demonstration. Record unavailable settings as unknown.

**The test harness is not a security sandbox.** Never paste or execute unreviewed model code, file-access commands, package installers, credentials, or network calls.

In [1]:
import sys
import platform
import pandas as pd

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("Execution mode: local Python teaching reference")
print("Bedrock calls: none; SAS execution: not performed")

Python: 3.13.5
pandas: 2.2.3
Execution mode: local Python teaching reference
Bedrock calls: none; SAS execution: not performed


## 1. Understand before translating

Read this SAS source. Before looking at the Python solution, explain the unit of calculation, how missing values affect it, and the output you expect for each row.

```sas
data test;
  input x y z;
  average = mean(x, y, z);
  datalines;
1 2 3
4 . 6
. . .
;
run;
proc print data=test noobs;
run;
```

SAS MEAN averages the nonmissing arguments; no observed arguments produce a missing result. These three expected results are therefore **2.0, 5.0, and missing**. They follow documented semantics; they are not captured SAS printouts. [SAS]

**Explain-only prompt:**
> Explain this SAS program without translating it. Identify inputs, output columns, transformations and dependencies. Separate source-supported behavior from assumptions. Flag missing definitions before describing affected behavior.

**How:** Review the explanation against the source. **Why:** Settle what the translation must preserve. **Why not:** Do not treat a fluent explanation as verified program behavior.

## 2. Specify the boundary

The candidate receives a pandas DataFrame with columns `case_id`, `row_id`, `x`, `y`, `z`. Only `x`, `y`, and `z` are measurements. The labels support the checks and must not be averaged. Measurement columns are `float64`; ordinary SAS numeric missing is represented here by NaN.

The candidate must return a new DataFrame, preserve the input unchanged, retain row order, index, labels, measurement values and types, and append one numeric column named `average`. It excludes missing measurements, keeps an all-missing average missing, and preserves zeros and negative numbers.

The interface and non-mutation requirements are explicit Python teaching requirements added around the SAS calculation. They are not claims about implicit SAS behavior.

**Out of scope:** special SAS missing codes, weighted or grouped means, character conversion, infinities, large-value numerical stability, statistical procedures, production data loading, and performance. Other numerical problems need agreed comparison tolerances; the chosen small fixture values use exact comparisons.

### Preserve expectations separately

The following shared fixture is copied from module 01 at the recorded repository commit. The first three rows are the original fixture. Two additional cases check zero and a lone negative value. `__MISSING__` is a storage sentinel decoded by the loader; it is never passed as a numeric value.

**Do not edit the expected values to match candidate output.** This notebook was built from the same snapshot stored in `notebook/fixtures.csv`, rather than inventing another set of test data. [Fixture; Project]

In [2]:
FIXTURE_CSV = '"case_id","row_id","x","y","z","expected_average"\r\n"fixture","f1","1","2","3","2"\r\n"fixture","f2","4","__MISSING__","6","5"\r\n"fixture","f3","__MISSING__","__MISSING__","__MISSING__","__MISSING__"\r\n"edge","e1","0","__MISSING__","__MISSING__","0"\r\n"edge","e2","__MISSING__","-6","__MISSING__","-6"\r\n'

"""Three named acceptance tests for the MEAN pilot, adapted from module 01.
Expected values must be supplied separately from the candidate calculation.
The callable runs ordinary Python. This harness is not a security sandbox.
"""
import io
import unittest
from typing import Callable, TextIO
import pandas as pd

INPUT_COLUMNS = ["case_id", "row_id", "x", "y", "z"]
MEASUREMENT_COLUMNS = ["x", "y", "z"]


def load_cases(csv_text: str) -> pd.DataFrame:
    """Decode the explicit missing sentinel; do not infer label missingness."""
    cases = pd.read_csv(io.StringIO(csv_text), dtype=str,
                        keep_default_na=False, na_filter=False)
    for col in MEASUREMENT_COLUMNS + ["expected_average"]:
        cases[col] = pd.to_numeric(cases[col].replace(
            "__MISSING__", float("nan")), errors="raise").astype("float64")
    if cases["row_id"].duplicated().any():
        raise ValueError("Fixture row_id values must be unique.")
    return cases


def run_acceptance(transform: Callable[[pd.DataFrame], pd.DataFrame],
                   cases: pd.DataFrame, stream: TextIO | None = None
                   ) -> unittest.TestResult:
    """Run T1, T2 and T3 against the supplied callable; return real results."""
    if not callable(transform):
        raise TypeError("transform must be a reviewed callable.")

    def check_values(actual: pd.DataFrame, expected_cases: pd.DataFrame) -> None:
        if not isinstance(actual, pd.DataFrame):
            raise AssertionError("The candidate must return a pandas DataFrame.")
        pd.testing.assert_series_equal(
            actual["average"].reset_index(drop=True),
            expected_cases["expected_average"].reset_index(drop=True),
            check_names=False, check_dtype=False, check_exact=True)

    class AcceptanceTests(unittest.TestCase):
        def test_T1_known_answers(self):
            subset = cases.loc[cases["case_id"].eq("fixture")].copy(deep=True)
            self.assertEqual(len(subset), 3)
            actual = transform(subset[INPUT_COLUMNS].copy(deep=True))
            check_values(actual, subset)

        def test_T2_discriminating_cases(self):
            subset = cases.loc[cases["case_id"].eq("edge")].copy(deep=True)
            self.assertEqual(len(subset), 2)
            actual = transform(subset[INPUT_COLUMNS].copy(deep=True))
            check_values(actual, subset)

        def test_T3_preservation(self):
            # Reversal and non-default index expose accidental reordering/reset.
            subset = cases.iloc[::-1].copy(deep=True)
            data = subset[INPUT_COLUMNS].copy(deep=True)
            data.index = [101 + 7 * i for i in range(len(data))]
            before = data.copy(deep=True)
            actual = transform(data)
            self.assertIsInstance(actual, pd.DataFrame)
            self.assertIsNot(actual, data)
            pd.testing.assert_frame_equal(data, before, check_exact=True)
            self.assertEqual(len(actual), len(before))
            self.assertTrue(actual.index.equals(before.index))
            self.assertEqual(actual.columns.tolist(),
                             before.columns.tolist() + ["average"])
            pd.testing.assert_frame_equal(actual[INPUT_COLUMNS], before,
                                          check_exact=True)
            self.assertTrue(pd.api.types.is_numeric_dtype(actual["average"]))
            self.assertFalse(pd.api.types.is_bool_dtype(actual["average"]))
            check_values(actual, subset)

    suite = unittest.defaultTestLoader.loadTestsFromTestCase(AcceptanceTests)
    return unittest.TextTestRunner(verbosity=2, stream=stream).run(suite)

cases = load_cases(FIXTURE_CSV)
cases

,case_id,row_id,x,y,z,expected_average
0,fixture,f1,1.0,2.0,3.0,2.0
1,fixture,f2,4.0,NaN,6.0,5.0
2,fixture,f3,NaN,NaN,NaN,NaN
3,edge,e1,0.0,NaN,NaN,0.0
4,edge,e2,NaN,-6.0,NaN,-6.0


## 3. Request a bounded translation

The prompts below are **new pilot adaptations**, not a replay of historical Bedrock trials. Attach or paste the same SAS source after each prompt. The shared interface makes both candidates callable by the same checks.

### Prompt A: leave implementation choices open

> Translate this SAS calculation into a Python function named `candidate(df)`. The input is a pandas DataFrame containing `case_id`, `row_id`, `x`, `y`, `z`. Return a new DataFrame that preserves the input columns, row order and index and appends numeric `average`; do not modify the input. Only `x`, `y`, `z` are measurements. Return code in the response. Do not install packages, read unrelated files, create files or execute code.

### Prompt B: make the consequential choices explicit

**Task**  
Translate only the row-wise SAS MEAN calculation into `candidate(df)`.

**Source and context**  
Use the attached SAS source. Input columns are `case_id`, `row_id`, `x`, `y`, `z`. Only `x`, `y`, `z` are measurements; the other columns are labels.

**Environment**  
Use Python with pandas. The three measurement columns are float64 and use NaN for ordinary numeric missing values. Use the approved environment, without installing packages.

**Required behavior**  
Average only `x`, `y`, `z` within each row, excluding missing values. Keep an all-missing average missing, preserve zeros and negatives, and return a new table without changing the input. Retain its columns, rows, row order and index; append numeric `average`.

**Deliverable**  
Return the function and a brief mapping to the requirements. Do not read unrelated files, create files, execute code, or introduce unrelated changes.

**Uncertainty and checks**  
Flag missing information or conflicts before implementing affected behavior. Do not invent a rule. For `[1,2,3]`, `[4,missing,6]`, and `[missing,missing,missing]`, expected averages are `2.0`, `5.0`, and missing. Check zero and a lone negative value as well. Keep expectations separate from the calculation. Label proposed checks as not executed.

**How:** Save the first response, then inspect its code separately. **Why:** Requirements and actual results can be compared. **Why not:** Do not promise that B must differ from A or that either prompt guarantees correctness.

### Trial record: fill this only for a real trial

| Field | Record |
|---|---|
| Trial ID and date | Not run in this package |
| Model identifier | Not recorded |
| Access application and visible settings | Not recorded; mark unavailable fields unknown |
| Source and prompt version | Pilot v0.1; source provenance below |
| Tool access and workspace isolation | Not recorded |
| First response location | Not captured |
| Execution environment and run output | Record after reviewed code is executed |

The local reference output below must not be copied into this table as a model response.

## 4. Inspect the candidate, then execute it

**The next cell contains the supplied teaching reference**, adapted from the existing module. It is not an observed model response. For a learner trial, preserve a copy of the workbook and replace only the `candidate` function with reviewed candidate code. Do not edit the fixture or test harness to obtain a pass.

The reference selects the measurement columns explicitly. `axis=1` computes across columns within each row, while `skipna=True` excludes missing values. [pandas]

In [3]:
"""Teaching reference adapted from module 01; not an observed model response."""
import pandas as pd


def candidate(df: pd.DataFrame) -> pd.DataFrame:
    """Append the row-wise mean without changing the supplied input table."""
    out = df.copy(deep=True)
    out["average"] = out[["x", "y", "z"]].mean(axis=1, skipna=True)
    return out

original_input = cases.loc[cases["case_id"].eq("fixture"), INPUT_COLUMNS].copy(deep=True)
reference_result = candidate(original_input)
reference_result

,case_id,row_id,x,y,z,average
0,fixture,f1,1.0,2.0,3.0,2.0
1,fixture,f2,4.0,NaN,6.0,5.0
2,fixture,f3,NaN,NaN,NaN,NaN


## 5. Run the acceptance checks

| Test | Evidence sought | Relevant mistake it can reject |
|---|---|---|
| T1: known answers | All original values and missing positions match | Filling missing values with zero or averaging down columns |
| T2: discriminating cases | Zero remains zero; a lone -6 remains -6 | Dropping observed zero or negative values |
| T3: preservation | Input unchanged; a new output preserves structure and values | Mutating the input, reordering rows, resetting the index, or dropping labels |

T3 reverses the fixture and uses a non-default index, so preserving only the original printed order is insufficient. Each named test contains multiple assertions. The helper compares to fixed expectations, not to another execution of the candidate. [Project; Tests]

A test failure is evidence to inspect, not a reason to weaken the expected result. Passing these cases does not prove general equivalence or authorize production use.

In [4]:
test_result = run_acceptance(candidate, cases, stream=sys.stdout)
if test_result.testsRun != 3 or not test_result.wasSuccessful():
    raise AssertionError("Acceptance checks failed. Inspect results before accepting the candidate.")
print("All three named acceptance tests passed for the callable just executed.")

test_T1_known_answers (__main__.run_acceptance.<locals>.AcceptanceTests.test_T1_known_answers) ... 

ok


test_T2_discriminating_cases (__main__.run_acceptance.<locals>.AcceptanceTests.test_T2_discriminating_cases) ... 

ok


test_T3_preservation (__main__.run_acceptance.<locals>.AcceptanceTests.test_T3_preservation) ... 

ok


----------------------------------------------------------------------
Ran 3 tests in 0.008s

OK


All three named acceptance tests passed for the callable just executed.


## 6. Record evidence and its limits

**Example decision for the supplied reference:**

| Record | Statement |
|---|---|
| Required behavior | Row-wise average of x, y, z excluding missing values, with the stated preservation requirements. |
| Evidence | Review the actual test output above and the recorded Python/pandas versions. T1, T2 and T3 each executed. |
| Limits | Only the five fixed cases and preservation checks were exercised. No SAS run or live Bedrock response is claimed; wider numerical and production behavior is untested. |
| Decision | Accept the supplied reference for this bounded teaching demonstration, subject to review. This is not production approval. |

For your own candidate, replace the decision with what your observed evidence supports. Also review explicit implementation and authorization requirements that these value/structure tests cannot establish.

**Reflection:** Which check would catch replacing missing values with zero? What would you need to establish before accepting the translation for a different dataset?

## Optional: diagnose a deliberately introduced error

This is an **instructor-authored negative example**, not a model failure or an A/B trial result. It intentionally fills missing values with zero before averaging. The exercise is disabled by default. Read the code, predict the mismatch, then set the flag to `True` to inspect it.

Draft a bounded repair request that includes the input, expected result, observed result and the affected code. Keep the original candidate and its acceptance results separate.

In [5]:
RUN_DIAGNOSTIC_EXAMPLE = False

if RUN_DIAGNOSTIC_EXAMPLE:
    incorrect = original_input.copy(deep=True)
    incorrect["average"] = incorrect[["x", "y", "z"]].fillna(0).mean(axis=1)
    print("Instructor-authored error, not an LLM response:")
    print(incorrect.to_string(index=False))
    print("For [4, missing, 6], expected 5.0; observed", incorrect["average"].iloc[1])
else:
    print("Optional diagnostic example not executed.")

Optional diagnostic example not executed.


## Source notes and provenance

Repository commit: `85c3041e74ddd7ef80086add2b29fa49292bebea`  
Fixture Git blob SHA-1: `fcd27f5fc32a142cf42fdf15d0f648780a1455de`

The fixture bytes are preserved from the repository. The core Python transformation is exposed as `candidate(df)`. The three-test harness is adapted to accept a callable and use explicit float64 inputs with exact comparisons for these small fixture values. The `.ipynb` embeds those components so a standalone copy has no dependency on an adjacent folder.

- **[SAS]** [SAS Institute: MEAN function](https://support.sas.com/documentation/cdl/en/lrdict/64316/HTML/default/a000245914.htm)
- **[pandas]** [pandas: DataFrame.mean](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.mean.html)
- **[Missing]** [pandas: Working with missing data](https://pandas.pydata.org/docs/user_guide/missing_data.html)
- **[Jupyter]** [Project Jupyter: The Jupyter Notebook](https://jupyter-notebook.readthedocs.io/en/stable/notebook.html)
- **[AWS]** [AWS: Inference parameters](https://docs.aws.amazon.com/bedrock/latest/userguide/inference-parameters.html)
- **[Project]** [SAS training modules: appendix design](https://github.com/the-pgh-cid/sas-training-modules/blob/85c3041e74ddd7ef80086add2b29fa49292bebea/APPENDICES.md)
- **[Fixture]** [Module 01: shared MEAN fixtures](https://github.com/the-pgh-cid/sas-training-modules/blob/85c3041e74ddd7ef80086add2b29fa49292bebea/module-01-mean/appendix/fixtures.csv)
- **[Reference]** [Module 01: Python reference](https://github.com/the-pgh-cid/sas-training-modules/blob/85c3041e74ddd7ef80086add2b29fa49292bebea/module-01-mean/appendix/reference.py)
- **[Tests]** [Module 01: Python acceptance tests](https://github.com/the-pgh-cid/sas-training-modules/blob/85c3041e74ddd7ef80086add2b29fa49292bebea/module-01-mean/appendix/test_reference.py)

Public documentation checked for this pilot on 25 September 2026. Follow the documentation for the actual versions used in your approved environment. The package's locally tested environment is recorded alongside the source files.